# Week 3 — Lab
## A full diagnostic suite for a classifier and a regressor

In this lab we put the theory to work with Yellowbrick, scikit-plot, and a few hand-rolled
Matplotlib figures. Two end-to-end workflows:

1. **Classifier:** breast-cancer Wisconsin → gradient-boosting → confusion matrix,
   ROC, PR, calibration, per-class report, learning curve.
2. **Regressor:** California housing → gradient-boosting → residuals, prediction error,
   feature importances, learning curve.

We will treat Yellowbrick as a productivity tool — fast for the standard charts — but
fall back to plain Matplotlib whenever the library's defaults are misleading or its
chart is missing a piece we need.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
)
from sklearn.calibration import calibration_curve

plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(0)


## Part A — Classifier diagnostics

Breast-cancer Wisconsin: 30 features, 2 classes (malignant / benign), 569 samples,
class balance ~63/37. Small but real.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(f"shape {X.shape},  positive rate {y.mean():.3f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    stratify=y, random_state=0)
clf = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
scores = clf.predict_proba(X_test)[:, 1]
preds = clf.predict(X_test)
print(f"test accuracy {clf.score(X_test, y_test):.3f}")


### A1 — Confusion matrix, row-normalized

The default confusion matrix shows counts. With imbalance, row-normalization (each row
sums to 1) gives **per-class recall** at a glance — and is what you usually want for
medical-style problems where "recall on the rare class" is the whole game.

In [ ]:
def confusion_matrix_plot(y_true, y_pred, labels, ax=None, normalize="row"):
    cm = confusion_matrix(y_true, y_pred)
    if normalize == "row":
        cm = cm / cm.sum(axis=1, keepdims=True)
        fmt = ".2f"
    else:
        fmt = "d"
    ax = ax or plt.gca()
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1 if normalize == "row" else cm.max())
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > (0.5 if normalize == "row" else cm.max() / 2) else "black"
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color=color, fontsize=11)
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    return im

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
confusion_matrix_plot(y_test, preds, data.target_names, ax=axes[0], normalize=None)
axes[0].set_title("Counts")
confusion_matrix_plot(y_test, preds, data.target_names, ax=axes[1], normalize="row")
axes[1].set_title("Row-normalized (per-class recall)")
plt.tight_layout(); plt.show()


### A2 — ROC, PR, calibration in one row

Yellowbrick has visualizers for each of these individually. For a compact diagnostic
panel we write a small helper that puts all three side by side — this is the chart that
should appear in any honest classifier report.

In [ ]:
def classifier_panel(y_true, scores, ax_roc=None, ax_pr=None, ax_cal=None):
    fig = plt.gcf() if (ax_roc and ax_pr and ax_cal) else None
    if fig is None:
        fig, (ax_roc, ax_pr, ax_cal) = plt.subplots(1, 3, figsize=(13, 4))

    # ROC
    fpr, tpr, _ = roc_curve(y_true, scores)
    ax_roc.plot(fpr, tpr, lw=2, label=f"AUC = {auc(fpr, tpr):.3f}")
    ax_roc.plot([0, 1], [0, 1], "k--", lw=0.7)
    ax_roc.set(xlabel="FPR", ylabel="TPR", title="ROC")
    ax_roc.legend(loc="lower right")

    # PR
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ax_pr.plot(rec, prec, lw=2,
               label=f"AP = {average_precision_score(y_true, scores):.3f}")
    ax_pr.axhline(np.mean(y_true), color="grey", ls="--", lw=0.7,
                  label=f"baseline = {np.mean(y_true):.2f}")
    ax_pr.set(xlabel="recall", ylabel="precision", title="Precision–Recall")
    ax_pr.legend(loc="lower left")

    # Calibration
    pp, pt = calibration_curve(y_true, scores, n_bins=10, strategy="quantile")
    ax_cal.plot(pp, pt, marker="o", lw=2)
    ax_cal.plot([0, 1], [0, 1], "k--", lw=0.7)
    ax_cal.set(xlabel="mean predicted prob.", ylabel="observed fraction positive",
               title="Calibration")
    plt.tight_layout()
    return fig

classifier_panel(y_test, scores)
plt.show()


### A3 — Per-class report as a table

`classification_report` is built into sklearn but its formatted-string output is hard to
read. Wrap it in a dataframe.

In [ ]:
rep = classification_report(y_test, preds, target_names=data.target_names,
                            output_dict=True, zero_division=0)
rep_df = pd.DataFrame(rep).T.round(3)
rep_df


### A4 — Learning curve

A learning curve shows training and validation score as a function of training set
size. The pattern of the gap diagnoses bias vs. variance:

- Gap shrinks toward zero as $n$ grows → **bias-dominated** (more data won't help much).
- Big persistent gap → **variance-dominated** (more data probably will help).


In [ ]:
sizes = np.linspace(0.1, 1.0, 8)
tr_sz, tr_sc, va_sc = learning_curve(
    GradientBoostingClassifier(random_state=0), X, y,
    train_sizes=sizes, cv=5, scoring="roc_auc", random_state=0, n_jobs=-1,
)
tr_mean, tr_std = tr_sc.mean(1), tr_sc.std(1)
va_mean, va_std = va_sc.mean(1), va_sc.std(1)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(tr_sz, tr_mean, label="train", color="#0072B2")
ax.fill_between(tr_sz, tr_mean - tr_std, tr_mean + tr_std, color="#0072B2", alpha=0.2)
ax.plot(tr_sz, va_mean, label="cv", color="#D55E00")
ax.fill_between(tr_sz, va_mean - va_std, va_mean + va_std, color="#D55E00", alpha=0.2)
ax.set(xlabel="training samples", ylabel="ROC AUC", title="Learning curve (5-fold CV)")
ax.legend(); plt.show()


On this dataset the curves have almost closed by $n = 400$ — adding samples beyond
that point won't buy you AUC, so a research budget would be better spent on features or
a different model class. **This is the kind of decision the chart makes obvious that a
summary statistic cannot.**

## Part B — Regressor diagnostics

California housing: 8 features, 20 640 samples, predicting median house value in
$100k units.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
print(X.shape, y.describe())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
reg = GradientBoostingRegressor(random_state=0, n_estimators=200).fit(X_train, y_train)
y_hat = reg.predict(X_test)
resid = y_test.values - y_hat
print(f"RMSE = {np.sqrt(((resid) ** 2).mean()):.3f}")


### B1 — Residuals and predicted-vs-actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(y_hat, resid, alpha=0.25, s=10)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set(xlabel="predicted", ylabel="residual",
            title=f"Residuals vs. fitted — mean={resid.mean():+.3f}")

lo = min(y_test.min(), y_hat.min())
hi = max(y_test.max(), y_hat.max())
axes[1].scatter(y_test, y_hat, alpha=0.25, s=10)
axes[1].plot([lo, hi], [lo, hi], "k--", lw=0.8)
axes[1].set(xlabel="actual", ylabel="predicted",
            title="Predicted vs. actual — note ceiling at $500k")
plt.tight_layout(); plt.show()


Two real-world patterns the charts surface:

- **Mild heteroskedasticity** — variance grows with the predicted value.
- **A hard cap at the target maximum** — many actuals are clipped at $500k in the data,
  which produces a horizontal band of underpredictions. RMSE alone gives you no hint
  this is happening; the chart screams it.

### B2 — Feature importances

Tree models expose `feature_importances_`. Plot them as a horizontal bar chart, sorted,
with bars wide enough to read the labels.

In [ ]:
imp = pd.Series(reg.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index, imp.values, color="#0072B2")
ax.set(xlabel="gini importance", title="GB feature importances")
plt.show()


**Caveat.** Gini importance is biased toward high-cardinality features. For more
robust importance estimates use **permutation importance** (also in scikit-learn):

```python
from sklearn.inspection import permutation_importance
result = permutation_importance(reg, X_test, y_test, n_repeats=10, random_state=0)
```

We will return to attribution and importance properly in week 5.

### B3 — Residual distribution + qqplot

Two views of the same residuals: their histogram and their quantile-quantile plot
against a normal. The qqplot is the cheapest test for "are my residuals approximately
normal?", which matters for confidence-interval-style downstream uses.

In [ ]:
from scipy import stats
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(resid, bins=60, color="#0072B2", alpha=0.85, edgecolor="white")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set(xlabel="residual", ylabel="count", title="Residual distribution")

stats.probplot(resid, dist="norm", plot=axes[1])
axes[1].set_title("QQ plot vs. normal")
axes[1].get_lines()[0].set_markerfacecolor("#0072B2")
axes[1].get_lines()[0].set_markeredgecolor("#0072B2")
axes[1].get_lines()[0].set_markersize(3)
axes[1].get_lines()[1].set_color("black")
plt.tight_layout(); plt.show()


**Reading.** The histogram is heavy-tailed and slightly skewed. The QQ plot
confirms it: deviations from the line at both ends mean the residual tails are heavier
than normal. This is fine for ranking-style use, but if you want 95 % prediction
intervals from these residuals, a normal-based interval will be too narrow.

## What to do differently in your own research

- **Always** look at the residual / boundary plot — not just the metric. The plot finds
  failure modes that the metric averages away.
- **Lead with PR** on imbalanced datasets, not ROC.
- Build the **three-panel classifier diagnostic** (ROC + PR + calibration) as a habit.
  It takes one helper function and answers most reviewer questions.
- Watch for **clipping** in the target — predicted-vs-actual scatters reveal it
  instantly.

### Next

Exercises in `exercises/`. Try them, then compare to the reference solutions.
